# 5 · A tiny melody generator

Train a GRU on real melodies from the Nottingham MIDI dataset. Then give it a short prompt and listen to its continuation.
The notebook uses monophonic note sequences. It does not model rhythm or harmony.

## Setup

In [104]:
import io
import math
import random
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import Audio, display

try:
    import pretty_midi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pretty_midi"])
    import pretty_midi

QUICK = True
SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("QUICK:", QUICK)

device: cuda
QUICK: True


## 1. Download the Nottingham MIDI files

Nottingham contains over 1,000 short folk melodies. We use the MIDI files from the cleaned public repository.
The loader keeps the notes from the instrument with the most events and ignores timing for this first model.

In [105]:
DATA_DIR = Path("nottingham_midi")
ZIP_URL = "https://github.com/jukedeck/nottingham-dataset/archive/refs/heads/master.zip"

if not DATA_DIR.exists():
    print("downloading Nottingham MIDI files...")
    archive = urllib.request.urlopen(ZIP_URL).read()
    with zipfile.ZipFile(io.BytesIO(archive)) as z:
        midi_names = [name for name in z.namelist() if "/MIDI/" in name and name.lower().endswith((".mid", ".midi"))]
        for name in midi_names:
            target = DATA_DIR / Path(name).name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(z.read(name))

midi_paths = sorted(DATA_DIR.glob("*.mid")) + sorted(DATA_DIR.glob("*.midi"))
if QUICK:
    midi_paths = midi_paths[:500]
print("MIDI files:", len(midi_paths))

def read_melody(path):
    midi = pretty_midi.PrettyMIDI(str(path))
    instrument = max(midi.instruments, key=lambda item: len(item.notes))
    notes = sorted(instrument.notes, key=lambda item: (item.start, item.pitch))
    return [note.pitch for note in notes if 36 <= note.pitch <= 96]


melodies = [melody for path in midi_paths if (melody := read_melody(path))]
pitch_values = sorted({pitch for melody in melodies for pitch in melody})
NOTE_NAMES = [pretty_midi.note_number_to_name(pitch) for pitch in pitch_values]
note_to_idx = {pitch: i for i, pitch in enumerate(pitch_values)}
idx_to_note = {i: pretty_midi.note_number_to_name(pitch) for i, pitch in enumerate(pitch_values)}
vocab_size = len(pitch_values)

print("melodies:", len(melodies), "| vocabulary:", vocab_size, "MIDI pitches")
print("example melody:", " ".join(pretty_midi.note_number_to_name(p) for p in melodies[0][:16]))

MIDI files: 500
melodies: 500 | vocabulary: 51 MIDI pitches
example melody: E5 D5 B4 A4 B4 C5 B4 G4 A4 E5 D5 B4 A4 B4 C5 B4


## 2. Build next-note examples

Each input window contains 32 notes. The target is the same window, shifted forward by one note.

In [106]:
SEQ_LEN = 32

def make_windows(sequences, seq_len=SEQ_LEN, stride=4):
    xs, ys = [], []
    for sequence in sequences:
        for start in range(0, len(sequence) - seq_len, stride):
            xs.append([note_to_idx[pitch] for pitch in sequence[start:start + seq_len]])
            ys.append([note_to_idx[pitch] for pitch in sequence[start + 1:start + seq_len + 1]])
    return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)


perm = torch.randperm(len(melodies)).tolist()
n_val = max(1, int(0.1 * len(perm)))
val_melodies = [melodies[i] for i in perm[:n_val]]
train_melodies = [melodies[i] for i in perm[n_val:]]
X_train, Y_train = make_windows(train_melodies)
X_val, Y_val = make_windows(val_melodies)

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=128, shuffle=True)
print("training windows:", len(X_train), "| validation windows:", len(X_val))
print("one input:", " ".join(idx_to_note[i] for i in X_train[0].tolist()))
print("target:    ", " ".join(idx_to_note[i] for i in Y_train[0].tolist()))

training windows: 19769 | validation windows: 1685
one input: B4 C5 D5 B4 C5 D5 B4 G4 E4 D4 E4 F#4 G4 D4 B4 C5 B4 A4 G4 D5 B4 C5 D5 B4 G4 E4 D4 E4 F#4 G4 D4 C5
target:     C5 D5 B4 C5 D5 B4 G4 E4 D4 E4 F#4 G4 D4 B4 C5 B4 A4 G4 D5 B4 C5 D5 B4 G4 E4 D4 E4 F#4 G4 D4 C5 B4


## 4. Train a GRU

The GRU predicts a note at each time step. We compare its validation accuracy with the most common next-note baseline.

In [ ]:
class NoteGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_size=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        states, _ = self.gru(embedded) # <- here we use all hidden states, not just the last one, many-to-many
        return self.output(states)


model = NoteGRU(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

with torch.no_grad():
    baseline = torch.bincount(Y_train.flatten(), minlength=vocab_size).float()
    baseline_acc = (Y_val == baseline.argmax()).float().mean().item()
print(f"most-common-note baseline: {baseline_acc:.3f}")

n_epochs = 12 if QUICK else 30
for epoch in range(1, n_epochs + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    if epoch == 1 or epoch % 3 == 0:
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for i in range(0, len(X_val), 256):
                xb = X_val[i:i + 256].to(device)
                yb = Y_val[i:i + 256].to(device)
                pred = model(xb).argmax(-1)
                correct += (pred == yb).sum().item()
                total += yb.numel()
        print(f"epoch {epoch:02d} | loss {total_loss / len(X_train):.3f} | "
              f"val next-note accuracy {correct / total:.3f}")

most-common-note baseline: 0.142
epoch 01 | loss 2.315 | val next-note accuracy 0.350
epoch 03 | loss 1.691 | val next-note accuracy 0.403
epoch 06 | loss 1.533 | val next-note accuracy 0.382
epoch 09 | loss 1.433 | val next-note accuracy 0.380
epoch 12 | loss 1.365 | val next-note accuracy 0.382


## 5. Generate and listen

Give the model a short seed. Temperature controls how strongly it favours high-scoring notes.
The audio uses one fixed duration per note, so it is a rough preview rather than the original rhythm.

In [108]:
@torch.no_grad()
def generate(seed_notes, length=24, temperature=0.8):
    model.eval()
    tokens = [note_to_idx[pretty_midi.note_name_to_number(note)] for note in seed_notes]
    result = list(tokens)
    for _ in range(length):
        context = torch.tensor([result[-SEQ_LEN:]], dtype=torch.long, device=device)
        logits = model(context)[0, -1] / max(temperature, 1e-5)
        next_token = torch.multinomial(torch.softmax(logits, dim=0), 1).item()
        result.append(next_token)
    return [idx_to_note[token] for token in result]


def notes_to_audio(notes, seconds_per_note=0.18, sample_rate=22_050):
    audio = []
    count = int(seconds_per_note * sample_rate)
    time = np.arange(count) / sample_rate
    fade = max(1, int(0.01 * sample_rate))
    envelope = np.ones(count, dtype=np.float32)
    envelope[:fade] = np.linspace(0, 1, fade)
    envelope[-fade:] = np.linspace(1, 0, fade)

    for note in notes:
        midi = pretty_midi.note_name_to_number(note)
        frequency = 440 * 2 ** ((midi - 69) / 12)
        tone = 0.25 * np.sin(2 * np.pi * frequency * time) * envelope
        audio.append(tone.astype(np.float32))
    return np.concatenate(audio)


seed = [pretty_midi.note_number_to_name(pitch) for pitch in melodies[0][:6]]
for temperature in (0.3, 0.8, 1.3):
    melody = generate(seed, length=24, temperature=temperature)
    print(f"temperature {temperature}: {' '.join(melody)}")
    display(Audio(notes_to_audio(melody), rate=22_050))

temperature 0.3: E5 D5 B4 A4 B4 C5 A4 A4 G4 A4 B4 G4 G4 A4 A4 E4 G4 A4 E4 E4 A4 B4 C5 E5 D5 C5 B4 A4 E4 E4


temperature 0.8: E5 D5 B4 A4 B4 C5 A4 C5 B4 D5 B4 D5 E5 A4 E5 E5 E5 A4 A4 B4 C5 D5 E5 D5 C5 B4 C5 D5 B4 B4


temperature 1.3: E5 D5 B4 A4 B4 C5 A4 D5 C5 B4 C5 A4 A5 A5 A4 G5 A5 B5 A5 A5 D5 G5 B5 A5 G5 F#5 E5 F#5 A5 G5
